# Fase 3C — RoBERTuito (IroSvA) — v7

**Qué es la v7.** Es la **continuación de la v6**, no un experimento nuevo. Usa exactamente el mismo código de la v6 (datos, grupos, tokenizador, `crear_modelo`, `ClasificadorTrainer`, argumentos de entrenamiento) y **lee** de la carpeta v6 la configuración final, la temperatura y las predicciones del test. Agrega lo que pidió la evaluación del artículo:

1. **Exportación del modelo final** (la v6 borraba los checkpoints): un modelo por variante, entrenado con todo el train, la configuración v6 y la semilla 42, guardado en formato Hugging Face con sus metadatos e instrucciones de carga → reproducibilidad.
2. **Verificación**: el modelo cargado desde el disco reproduce la semilla 42 del test de la v6.
3. **Interpretabilidad con SHAP**: tokens que más influyen en la predicción por variante (análisis global) y ejemplos de aciertos y errores (análisis local).

**Lo que NO hace:** no repite la Nested CV, Optuna ni el test de 5 semillas, y no cambia ningún resultado. Los resultados oficiales de RoBERTuito siguen siendo los de la v6. Tiempo aproximado con GPU: 10–20 min de entrenamiento y 10–20 min de SHAP por variante.

**Requisitos en esta máquina:** la carpeta `../data/resultados_robertuito_grouped_5x3_tpe20_v6/` (al menos `robertuito_config_final.json`, `robertuito_temperatura.json` y `robertuito_test_probabilidades.csv`), los CSV de `../data/transformers/` y el paquete `shap` (`py -m uv add shap`). Resultados en `../data/resultados_robertuito_grouped_5x3_tpe20_v7/`.

## 1. Librerías y entorno

En la terminal del proyecto (mismo entorno que usa Jupyter):

```powershell
py -m uv add torch transformers accelerate optuna scikit-learn pandas numpy ipython
```

Si ya tienes las dependencias, no necesitas reinstalarlas. Este notebook **no instala paquetes ni inicia GPU** por sí solo al abrirlo.

In [ ]:
import os
import re

# Declarar antes de importar Transformers/Hugging Face para ocultar barras de descarga.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

import gc
import time
import json
import pickle
import html
import random
import tempfile
import shutil
import inspect
import warnings
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import optuna

from torch.utils.data import Dataset, Subset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from IPython.display import display, HTML

import matplotlib

matplotlib.use("Agg")  # Evita abrir una ventana por gráfica durante cientos de fits.
import matplotlib.pyplot as plt

import transformers
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    set_seed,
)
from transformers.trainer_callback import PrinterCallback, ProgressCallback
from transformers.trainer_utils import get_last_checkpoint

try:
    from huggingface_hub.utils import disable_progress_bars

    disable_progress_bars()
except (ImportError, AttributeError):
    pass
transformers.logging.set_verbosity_error()
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

VERSIONES = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "optuna": optuna.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
print("Versiones:", " | ".join(f"{k} {v}" for k, v in VERSIONES.items()))
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 2. Configuración

In [ ]:
MODEL_NAME = "pysentimiento/robertuito-base-uncased"

DATA_DIR = Path("../data/transformers")
# v7: la v6 se LEE (configuración final, temperatura y predicciones); la v7 escribe en su carpeta.
RESULTS_DIR_V6 = Path("../data/resultados_robertuito_grouped_5x3_tpe20_v6")
RESULTS_DIR = Path("../data/resultados_robertuito_grouped_5x3_tpe20_v7")
PROGRESS_DIR = RESULTS_DIR / "seguimiento"

TEXT_COLS = {"PYSENTIMIENTO": "MESSAGE_TRANSFORMER"}
LABEL_COL = "IS_IRONIC"
ID_COL = "ID"
ORIGINAL_COL = "MESSAGE"
GROUP_COL = "GROUP_ID"

MAX_LENGTH = 128
SEED = 42

# ------------------------------------------------------------
# "prueba" valida rápidamente todo el pipeline.
# "completo" = experimento definitivo 5×3 / TPE20.
# ------------------------------------------------------------
MODO = "completo"  # primero "prueba"; luego "completo" y reiniciar kernel

if MODO == "prueba":
    VARIANTES = ["mx"]
    N_OUTER = 2
    N_INNER = 2
    N_TRIALS = 2
    N_STARTUP_TRIALS = 1
elif MODO == "completo":
    VARIANTES = ["mx", "es", "cu"]
    N_OUTER = 5
    N_INNER = 3
    N_TRIALS = 20
    N_STARTUP_TRIALS = 5
else:
    raise ValueError("MODO debe ser 'prueba' o 'completo'.")

# RUN_TAG corto + subcarpetas compactas para Windows.
# v7: carpeta nueva; la v6 queda intacta.
RUN_TAG = "r5x3t20_v7"
RUN_DIR = PROGRESS_DIR / RUN_TAG / MODO
TMP_DIR = RUN_DIR / "tmp"
CHECKPOINT_DIR = RUN_DIR / "ckpt"
FIT_STATE_DIR = RUN_DIR / "state"
CURVE_DIR = RUN_DIR / "curves"
GRAPH_DIR = RUN_DIR / "graphs"
AUDIT_DIR = RUN_DIR / "audit"

for carpeta in [
    RESULTS_DIR,
    PROGRESS_DIR,
    RUN_DIR,
    TMP_DIR,
    CHECKPOINT_DIR,
    FIT_STATE_DIR,
    CURVE_DIR,
    GRAPH_DIR,
    AUDIT_DIR,
]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Flujo recomendado (sección 18): opcionalmente una corrida corta con MODO="prueba";
# luego MODO="completo". Nested CV y optimización final usan solo train y corren seguidas;
# el test oficial requiere además CONFIRMACION_TEST.
EJECUTAR_NESTED = False  # v7: la Nested CV ya está en la v6
EJECUTAR_OPTIMIZACION_FINAL = False  # v7: se lee la de la v6  # solo train; se ejecuta tras la Nested CV
EJECUTAR_TEST_OFICIAL = False
CONFIRMACION_TEST = ""

SEMILLAS_FINALES = [42, 52, 62, 72, 82]  # test v6 (no se repite en la v7)

# v7 — exportación del modelo final e interpretabilidad (SHAP)
SEMILLA_EXPORTACION = 42  # misma semilla que el primer modelo del test de la v6
N_SHAP_POR_TIPO = 50  # tuits explicados por tipo (VP, VN, FP, FN) y variante
SHAP_BATCH = 32  # lote de inferencia de SHAP (bajar si falta memoria de GPU)
MIN_TEXTOS_TOKEN = 5  # un token entra al análisis global si aparece en ≥ 5 tuits explicados
TOP_TOKENS = 10  # tokens por sentido (irónico / no irónico) en la figura global
MAX_TOKENS_EJEMPLO = 40  # tokens mostrados en cada ejemplo local
CLEAN_COMPLETED_CHECKPOINTS = True

# Seguimiento: panel de progreso común a todos los notebooks de modelos.
NOMBRE_EXPERIMENTO = "RoBERTuito"
HEARTBEAT_SEGUNDOS = 30
# Pitido de Windows al terminar o detenerse una etapa larga (False para desactivarlo).
AVISO_SONORO = True

# Optuna TPE: 5 trials iniciales aleatorios y 15 adaptativos en modo completo.
# Espacio común a BETO, DistilBETO y RoBERTuito.
SEARCH_SPACE = {
    "preprocessing": ["PYSENTIMIENTO"],
    "learning_rate": [2e-5],  # fijo: elegido en 45/45 Outer Folds de la v4
    "batch_size": [16],  # fijo: pilotos B y C
    "num_train_epochs": [6],  # fijo: máximo elegido en 44/45 Outer Folds; decide el early stopping
    "weight_decay": [0.01, 0.1],
    "loss_mode": ["balanced"],  # fijo: elegido en 37/45 Outer Folds de la v4
    "classifier_dropout": [0.1, 0.2, 0.3],
    "label_smoothing": [0.1],  # fijo: piloto P3 (sin LS la NLL de validación vuelve a subir)
    "lr_decay": [0.9],  # fijo (piloto B)
    # v6 — regularización FIJA por modelo, elegida con los pilotos v5b (rondas 2–5) y la regla
    # fijada antes de verlos: "la regularización más fuerte que no baje el F1 de validación > 0.01".
    # RoBERTuito: solo dropout 0.2 (piloto P12); enmascarar + dropout (P19) bajó el F1 de RoBERTuito.
    # (En la v5 estaban en la búsqueda y Optuna, que elige por F1, casi no las usó.)
    "dropout_interno": [0.2],  # fijo: P12 pasó en los 6 casos; 0.3 (P18) bajó el F1
    "congelar_frac": [0.0],  # fijo: sin congelar (P13 = congelar + dropout no superó a P12)
    "mascara_tokens": [0.0],  # fijo: % de tokens reemplazados por [MASK] SOLO al entrenar (P16/P19)
}

# Warmup EXPLÍCITO. En transformers 5.x `warmup_ratio` ya no existe y la v4 lo descartaba en
# silencio (la v4 corrió sin warmup). El piloto P9 mostró que el warmup del 10 % no cambia el F1
# ni la NLL, así que la v5 mantiene 0.0 por continuidad con la v4, ahora declarado y verificado.
WARMUP_FRACCION = 0.0

# Early stopping de los Inner Folds: épocas sin mejora de F1 antes de detener.
PACIENCIA_EARLY_STOPPING = 2

SEARCH_SPACE_REQUERIDO = {
    "preprocessing",
    "learning_rate",
    "batch_size",
    "num_train_epochs",
    "weight_decay",
    "loss_mode",
    "classifier_dropout",
    "label_smoothing",
    "lr_decay",
    "dropout_interno",
    "congelar_frac",
    "mascara_tokens",
}
faltantes_search_space = SEARCH_SPACE_REQUERIDO - set(SEARCH_SPACE)
if faltantes_search_space:
    raise KeyError(f"SEARCH_SPACE incompleto. Faltan: {sorted(faltantes_search_space)}")

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

fits_teoricos = len(VARIANTES) * N_OUTER * (N_TRIALS * N_INNER + 1)

print(f"[CONFIG][RoBERTuito] modo={MODO} | variantes={VARIANTES}")
print(
    f"[CONFIG][RoBERTuito] Nested={N_OUTER}×{N_INNER} | "
    f"trials={N_TRIALS} | startup={N_STARTUP_TRIALS}"
)
print(
    f"[CONFIG][RoBERTuito] fine-tunings teóricos={fits_teoricos} | "
    f"nested_activo={EJECUTAR_NESTED}"
)
print(f"[CONFIG][RoBERTuito] RUN_DIR={RUN_DIR.resolve()}")
print("[CONFIG][RoBERTuito] SEARCH_SPACE OK:", sorted(SEARCH_SPACE))

# Comprobación preventiva de longitud de ruta en Windows (peor caso del espacio).
_ruta_ckpt_ejemplo = (
    CHECKPOINT_DIR / "mx" / "outer1" / "P_l2_b16_e6_w10_b_d3_s1_r9_i2_c25_m15" / "inner3" / "checkpoint-99999"
).resolve()

print(
    f"[CONFIG][RoBERTuito] longitud ruta checkpoint ejemplo="
    f"{len(str(_ruta_ckpt_ejemplo))} caracteres"
)

if os.name == "nt" and len(str(_ruta_ckpt_ejemplo)) >= 230:
    raise RuntimeError(
        "La ruta prevista para checkpoints sigue siendo demasiado larga en Windows. "
        "Mueve el repositorio a una ruta más corta antes de entrenar."
    )

## 3. Carga del train y verificación del preprocesamiento

BETO, DistilBETO y RoBERTuito consumen los mismos CSV finales.

Antes de construir grupos se verifica:

- manifest del preprocesamiento y versión del pipeline;
- número final de registros por variante;
- IDs únicos;
- etiquetas binarias;
- `MESSAGE_TRANSFORMER` no vacío.

Este notebook no retira registros del train en función del test.

In [ ]:
MANIFEST_PATH = DATA_DIR / "manifest_preprocesamiento_transformers.csv"
PIPELINE_VERSION_ESPERADA = "transformers_comun_v2"


def cargar_manifest_preprocesamiento():
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"Falta {MANIFEST_PATH}. Ejecuta primero "
            "`01_preprocesamiento_dl_transformers.ipynb`."
        )

    manifest = pd.read_csv(MANIFEST_PATH)

    requeridas = {"pipeline_version", "variante", "train_final"}
    faltantes = requeridas - set(manifest.columns)
    if faltantes:
        raise ValueError(f"Manifest incompleto; faltan columnas: {sorted(faltantes)}")

    versiones = set(manifest["pipeline_version"].astype(str))
    if versiones != {PIPELINE_VERSION_ESPERADA}:
        raise ValueError(
            f"Versión de preprocessing inesperada: {versiones}. "
            f"Se esperaba {PIPELINE_VERSION_ESPERADA}."
        )

    return manifest


MANIFEST_PREPROCESAMIENTO = cargar_manifest_preprocesamiento()


def validar_archivo_train_con_manifest(variante):
    ruta = DATA_DIR / f"train_transformers_{variante}.csv"
    if not ruta.exists():
        raise FileNotFoundError(f"Falta {ruta}. Ejecuta primero el preprocessing Transformer.")

    fila = MANIFEST_PREPROCESAMIENTO[MANIFEST_PREPROCESAMIENTO["variante"].astype(str) == variante]
    if len(fila) != 1:
        raise ValueError(f"{variante}: se esperaba exactamente una fila en el manifest.")

    return ruta, fila.iloc[0]


def cargar_train(variante):
    ruta, manifest_row = validar_archivo_train_con_manifest(variante)
    df = pd.read_csv(ruta)

    requeridas = [ID_COL, ORIGINAL_COL, LABEL_COL, *TEXT_COLS.values()]
    faltantes = [c for c in requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(f"{variante}: faltan columnas {faltantes}")

    df[ID_COL] = df[ID_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    for col in [ORIGINAL_COL, *TEXT_COLS.values()]:
        df[col] = df[col].fillna("").astype(str)

    if df[ID_COL].duplicated().any():
        raise ValueError(f"{variante}: IDs duplicados en el train FINAL.")

    if not set(df[LABEL_COL].unique()).issubset({0, 1}):
        raise ValueError(f"{variante}: etiquetas fuera de 0/1.")

    vacios = {
        col: int(df[col].eq("").sum())
        for col in [ORIGINAL_COL, *TEXT_COLS.values()]
        if df[col].eq("").any()
    }
    if vacios:
        raise ValueError(
            f"{variante}: entradas vacías en el train final {vacios}. "
            "Vuelve a ejecutar `01_preprocesamiento_dl_transformers.ipynb` "
            "(MESSAGE_TRANSFORMER requiere `pysentimiento` instalado)."
        )

    esperado = int(manifest_row["train_final"])
    if len(df) != esperado:
        raise ValueError(f"{variante}: train={len(df)} pero manifest={esperado}.")

    return df.reset_index(drop=True)


for variante in ["mx", "es", "cu"]:
    tmp = cargar_train(variante)
    print(
        f"[DATA][RoBERTuito][{variante.upper()}] "
        f"n={len(tmp)} | "
        f"clases={tmp[LABEL_COL].value_counts().sort_index().to_dict()} | "
        "manifest=OK"
    )

## 4. Tokenizador oficial y datos

Se usa el **tokenizer propio de RoBERTuito** (uncased). Los tres Transformers reciben el mismo texto, `MESSAGE_TRANSFORMER`, generado con `pysentimiento.preprocessing.preprocess_tweet`. No se elimina puntuación, tildes ni stopwords. El texto se tokeniza previamente (sin usar etiquetas para crear el vocabulario) y cada partición utiliza índices del mismo dataframe. Padding dinámico por batch.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Tokenizer:", tokenizer.__class__.__name__)
print("do_lower_case:", getattr(tokenizer, "do_lower_case", "gestionado por tokenizer"))
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None
)


class DatasetTokenizado(Dataset):
    def __init__(self, textos, etiquetas):
        self.encoded = tokenizer(
            [str(t) for t in textos], truncation=True, max_length=MAX_LENGTH, padding=False
        )
        self.labels = np.asarray(etiquetas, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encoded.items()}
        item["labels"] = int(self.labels[i])
        return item


def cache_dataset(df):
    y = df[LABEL_COL].to_numpy(dtype=np.int64)
    return {prep: DatasetTokenizado(df[col].tolist(), y) for prep, col in TEXT_COLS.items()}, y

### 4.1 Auditoría y construcción de `GROUP_ID`

RoBERTuito crea sus grupos **únicamente para impedir leakage entre folds** dentro del train.

Se consideran equivalentes:

- texto original;
- `MESSAGE_TRANSFORMER`;
- secuencia tokenizada de `MESSAGE_TRANSFORMER` con el tokenizer de RoBERTuito.

Los grupos son propios de RoBERTuito y no dependen de los grupos o splits de los otros modelos.

In [ ]:
from collections import defaultdict


def colapsar_espacios(texto):
    return re.sub(r"\s+", " ", str(texto)).strip()


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def clave_tokens(ids):
    """Representación directa de los input_ids, sin hashes."""
    return ",".join(map(str, ids))


def construir_grupos_robertuito(df, variante, tokenizador=tokenizer, max_length=MAX_LENGTH):
    """Agrupa entradas equivalentes para evitar que crucen train/validación.

    Agrupación propia del modelo: texto original, cada representación de
    entrada (TEXT_COLS) y su secuencia tokenizada (truncada a MAX_LENGTH).
    """
    trabajo = df.copy().reset_index(drop=True)
    trabajo["KEY_ORIGINAL"] = trabajo[ORIGINAL_COL].map(colapsar_espacios)
    for prep, columna in TEXT_COLS.items():
        trabajo[f"KEY_{prep}"] = trabajo[columna].map(colapsar_espacios)

    for prep, columna in TEXT_COLS.items():
        ids = tokenizador(
            trabajo[columna].astype(str).tolist(),
            truncation=True,
            max_length=max_length,
            padding=False,
        )["input_ids"]
        trabajo[f"KEY_TOK_{prep}"] = [clave_tokens(x) for x in ids]

    representaciones = ["ORIGINAL", *TEXT_COLS, *[f"TOK_{p}" for p in TEXT_COLS]]
    key_cols = [f"KEY_{r}" for r in representaciones]
    uf = UnionFind(len(trabajo))

    for col in key_cols:
        first_seen = {}
        for idx, key in enumerate(trabajo[col].tolist()):
            if not key:
                continue
            if key in first_seen:
                uf.union(idx, first_seen[key])
            else:
                first_seen[key] = idx

    componentes = defaultdict(list)
    for idx in range(len(trabajo)):
        componentes[uf.find(idx)].append(idx)

    group_ids = [None] * len(trabajo)

    componentes_ordenados = sorted(componentes.values(), key=lambda miembros: min(miembros))

    for numero_grupo, miembros in enumerate(componentes_ordenados, start=1):
        gid = f"{variante}_G{numero_grupo:04d}"
        for idx in miembros:
            group_ids[idx] = gid
    trabajo[GROUP_COL] = group_ids

    resumen = []
    for nombre, col in zip(representaciones, key_cols):
        sizes = trabajo.groupby(col).size()
        repetidos = sizes[sizes > 1]
        resumen.append(
            {
                "variante": variante,
                "representacion": nombre,
                "filas_en_repeticiones": int(trabajo[col].isin(repetidos.index).sum()),
                "grupos_repetidos": int(len(repetidos)),
            }
        )

    gstats = trabajo.groupby(GROUP_COL)[LABEL_COL].agg(["size", "nunique"])
    conflictivos = set(gstats[(gstats["size"] > 1) & (gstats["nunique"] > 1)].index)
    detalle_conflictos = trabajo[trabajo[GROUP_COL].isin(conflictivos)].copy()

    resumen = pd.DataFrame(resumen)
    resumen["n_train"] = len(trabajo)
    resumen["n_group_id"] = trabajo[GROUP_COL].nunique()
    resumen["max_group_size"] = int(trabajo.groupby(GROUP_COL).size().max())
    resumen["grupos_con_labels_mixtos"] = len(conflictivos)
    return trabajo, resumen, detalle_conflictos


TRAIN_AGRUPADO = {}
AUDITORIA_GRUPOS = []
DETALLES_CONFLICTOS = []

# Se construyen los grupos de las tres variantes antes de una corrida larga.
for variante in ["mx", "es", "cu"]:
    df_audit = cargar_train(variante)
    agrupado, resumen, conflictos = construir_grupos_robertuito(df_audit, variante)
    TRAIN_AGRUPADO[variante] = agrupado
    AUDITORIA_GRUPOS.append(resumen)
    if not conflictos.empty:
        conflictos["variante"] = variante
        DETALLES_CONFLICTOS.append(conflictos)

df_auditoria_grupos = pd.concat(AUDITORIA_GRUPOS, ignore_index=True)
df_auditoria_grupos.to_csv(RUN_DIR / "auditoria_grupos_train.csv", index=False)
display(df_auditoria_grupos)

if DETALLES_CONFLICTOS:
    df_conflictos_grupo = pd.concat(DETALLES_CONFLICTOS, ignore_index=True)
    df_conflictos_grupo.to_csv(RUN_DIR / "auditoria_grupos_labels_mixtos.csv", index=False)

for variante in ["mx", "es", "cu"]:
    df_g = TRAIN_AGRUPADO[variante]
    n_mixtos = int(df_g.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum())
    print(
        f"[AUDIT][RoBERTuito][{variante.upper()}] n={len(df_g)} | "
        f"groups={df_g[GROUP_COL].nunique()} | "
        f"max_group={df_g.groupby(GROUP_COL).size().max()} | labels_mixtos={n_mixtos}"
    )

    if n_mixtos > 0:
        print(
            f"  ADVERTENCIA: {n_mixtos} GROUP_ID contienen etiquetas mixtas. "
            "No es data leakage porque el grupo permanece íntegro dentro de un fold, "
            "pero representa ambigüedad/ruido de etiqueta y debe documentarse."
        )

### 4.2 Pre-flight metodológico

El pre-flight valida el train final y los grupos de RoBERTuito.

No accede al test para eliminar o decidir registros.

In [ ]:
def comprobar_preflight_robertuito():
    print("=" * 78)
    print("PRE-FLIGHT — ROBERTUITO GROUPED NESTED CV")
    print("=" * 78)

    if MODO == "completo":
        esperado = (N_OUTER, N_INNER, N_TRIALS, N_STARTUP_TRIALS)

        if esperado != (5, 3, 20, 5):
            raise ValueError(
                f"Configuración inesperada: {esperado}; " "se esperaba 5×3, 20 trials, startup=5."
            )

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA no disponible. Verifica entorno/PyTorch antes de entrenar RoBERTuito."
        )

    for variante in ["mx", "es", "cu"]:
        _, manifest_row = validar_archivo_train_con_manifest(variante)

        df = TRAIN_AGRUPADO[variante]

        if len(df) != int(manifest_row["train_final"]):
            raise ValueError(f"{variante}: train agrupado no coincide " "con el manifest.")

        if df[ID_COL].duplicated().any():
            raise ValueError(f"{variante}: IDs duplicados.")

        if df[GROUP_COL].isna().any():
            raise ValueError(f"{variante}: GROUP_ID nulo.")

        if not set(df[LABEL_COL].unique()).issubset({0, 1}):
            raise ValueError(f"{variante}: etiquetas fuera de 0/1.")

        n_mixtos = int(df.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum())

        if n_mixtos:
            print(
                f"[AUDIT][RoBERTuito][{variante.upper()}] "
                f"ADVERTENCIA: {n_mixtos} GROUP_ID "
                "con etiquetas mixtas; se documentan "
                "como ruido/ambigüedad."
            )

    print("Preprocessing:", PIPELINE_VERSION_ESPERADA)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Nested: {N_OUTER}×{N_INNER} | Optuna={N_TRIALS} | startup={N_STARTUP_TRIALS}")
    print("Train final:", {v: len(TRAIN_AGRUPADO[v]) for v in ["mx", "es", "cu"]})
    print("PRE-FLIGHT DE DATOS: OK")


comprobar_preflight_robertuito()

## 5. Métricas, pesos de clase y modelos independientes

En cada Inner Fold la pérdida balanceada calcula pesos **solo** con el train de ese fold. Cada llamada `crear_modelo()` carga RoBERTuito desde cero con nueva cabeza de clasificación binaria. Los mensajes de pesos `MISSING`/`UNEXPECTED` quedan en un log, no en Jupyter.

**v4.** `ClasificadorTrainer` aplica label smoothing solo durante el entrenamiento (la pérdida de validación sigue siendo entropía cruzada estándar) y un learning rate decreciente por capas: la cabeza usa el LR base y cada capa inferior lo multiplica por 0.9. `ParadaTemprana` hace early stopping sin guardar checkpoints y `DetenerEnEpoca` detiene el refit en la época elegida sin alterar el scheduler.

In [ ]:
def establecer_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def softmax_np(logits):
    z = np.asarray(logits, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def nll_binaria(probs, labels):
    """NLL estándar: SIN pesos de clase y SIN label smoothing (auditoría v5)."""
    labels = np.asarray(labels, dtype=int)
    return float(-np.mean(np.log(np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0))))


def ece_binaria(probs, labels, n_bins=10):
    """ECE con 10 bins de igual masa sobre la confianza de la clase predicha."""
    labels = np.asarray(labels, dtype=int)
    conf = probs.max(axis=1)
    acierto = (probs.argmax(axis=1) == labels).astype(float)
    grupos = np.array_split(np.argsort(conf, kind="stable"), n_bins)
    return float(sum(len(g) / len(conf) * abs(acierto[g].mean() - conf[g].mean())
                     for g in grupos if len(g)))


def metricas(prediction):
    logits, labels = prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    y_pred = np.asarray(logits).argmax(axis=-1)
    probs = softmax_np(logits)
    return {
        "nll": nll_binaria(probs, labels),
        "ece": ece_binaria(probs, labels),
        "f1_macro": f1_score(labels, y_pred, average="macro"),
        "accuracy": accuracy_score(labels, y_pred),
        "precision_macro": precision_score(labels, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, y_pred, average="macro", zero_division=0),
    }


def pesos_balanceados(y):
    classes = np.array([0, 1])
    w = compute_class_weight("balanced", classes=classes, y=np.asarray(y))
    return torch.tensor(w, dtype=torch.float32)


PATRON_CAPA = re.compile(r"\.layer\.(\d+)\.")


def congelar_fraccion_inferior(model, fraccion):
    """v5: congela embeddings + la fracción inferior de capas del encoder (piloto P1)."""
    if not fraccion:
        return []
    n_total = 1 + max(int(m.group(1)) for n, _ in model.named_parameters()
                      if (m := PATRON_CAPA.search(n)))
    n_congeladas = max(1, round(n_total * float(fraccion)))
    capas = set()
    for nombre, p in model.named_parameters():
        m = PATRON_CAPA.search(nombre)
        if "embeddings" in nombre or (m and int(m.group(1)) < n_congeladas):
            p.requires_grad = False
            if m:
                capas.add(int(m.group(1)))
    return sorted(capas)


def crear_modelo(seed, classifier_dropout=0.2, cfg=None):
    """v5: `cfg` aporta `dropout_interno` (piloto P12) y `congelar_frac` (piloto P1)."""
    cfg = cfg or {}
    establecer_semilla(seed)
    with (RUN_DIR / "carga_modelos.log").open("a", encoding="utf-8") as log, redirect_stdout(
        log
    ), redirect_stderr(log):
        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
        if config.model_type != "roberta":
            raise ValueError(f"Se esperaba arquitectura RoBERTa, recibida: {config.model_type}")
        # RoBERTa: dropout de la cabeza de clasificación.
        config.classifier_dropout = float(classifier_dropout)
        # v5: dropout interno (oculto y de atención) del encoder y los embeddings.
        dropout_interno = float(cfg.get("dropout_interno", getattr(config, "hidden_dropout_prob")))
        config.hidden_dropout_prob = dropout_interno
        config.attention_probs_dropout_prob = dropout_interno
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
    congelar_fraccion_inferior(model, cfg.get("congelar_frac", 0.0))
    return model


class ClasificadorTrainer(Trainer):
    """v6: pérdida ponderada opcional + label smoothing (solo en train) + LR decay por capas
    + enmascarado aleatorio de tokens (solo en train; piloto P16/P19)."""

    def __init__(self, *args, class_weights=None, label_smoothing=0.0, lr_decay=1.0,
                 mascara_tokens=0.0, **kwargs):
        self.class_weights = class_weights
        self.label_smoothing = float(label_smoothing)
        self.lr_decay = float(lr_decay)
        self.mascara_tokens = float(mascara_tokens)
        self._especiales = torch.tensor(sorted(set(tokenizer.all_special_ids)))
        super().__init__(*args, **kwargs)

    def _enmascarar(self, inputs):
        """Reemplaza al azar `mascara_tokens` de los tokens (no especiales, no padding) por el
        token de máscara. Validación, refit-evaluación y test leen siempre el texto completo."""
        ids = inputs["input_ids"].clone()
        candidatos = ~torch.isin(ids, self._especiales.to(ids.device))
        if "attention_mask" in inputs:
            candidatos &= inputs["attention_mask"].bool()
        sorteo = torch.rand(ids.shape, device=ids.device) < self.mascara_tokens
        ids[candidatos & sorteo] = tokenizer.mask_token_id
        return {**inputs, "input_ids": ids}

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        if model.training and self.mascara_tokens > 0:
            inputs = self._enmascarar(inputs)
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        weights = (
            self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        )
        # Smoothing SOLO al entrenar: la pérdida de validación sigue siendo entropía cruzada estándar.
        ls = self.label_smoothing if model.training else 0.0
        loss = F.cross_entropy(outputs.logits, labels, weight=weights, label_smoothing=ls)
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self, *args, **kwargs):
        if self.optimizer is not None:
            return self.optimizer
        if self.lr_decay >= 1.0:
            return super().create_optimizer(*args, **kwargs)
        # Capas: BERT/RoBERTa `encoder.layer.N`, DistilBERT `transformer.layer.N`.
        patron = re.compile(r"\.layer\.(\d+)\.")
        n_capas = 1 + max(
            int(m.group(1)) for n, _ in self.model.named_parameters() if (m := patron.search(n))
        )
        base_lr, wd = self.args.learning_rate, self.args.weight_decay
        grupos = {}
        for nombre, p in self.model.named_parameters():
            if not p.requires_grad:
                continue
            m = patron.search(nombre)
            if m:
                profundidad = n_capas - int(m.group(1))  # última capa = 1
            elif "embeddings" in nombre:
                profundidad = n_capas + 1  # embeddings = LR más bajo
            else:
                profundidad = 0  # cabeza de clasificación / pooler = LR base
            sin_wd = nombre.endswith("bias") or "LayerNorm" in nombre or "layer_norm" in nombre
            clave = (base_lr * self.lr_decay**profundidad, 0.0 if sin_wd else wd)
            grupos.setdefault(clave, []).append(p)
        self.optimizer = torch.optim.AdamW(
            [{"params": ps, "lr": lr, "weight_decay": w} for (lr, w), ps in grupos.items()]
        )
        return self.optimizer


class ParadaTemprana(TrainerCallback):
    """Early stopping por F1-Macro de validación sin guardar checkpoints (Inner Folds)."""

    def __init__(self, paciencia=2):
        self.paciencia, self.mejor, self.sin_mejora = int(paciencia), -np.inf, 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        f1 = (metrics or {}).get("eval_f1_macro")
        if f1 is None:
            return control
        if f1 > self.mejor:
            self.mejor, self.sin_mejora = f1, 0
        else:
            self.sin_mejora += 1
            if self.sin_mejora >= self.paciencia:
                control.should_training_stop = True
        return control


class DetenerEnEpoca(TrainerCallback):
    """Refit/test con el MISMO scheduler que la búsqueda: se detiene en la época elegida."""

    def __init__(self, epocas):
        self.epocas = int(epocas)

    def on_epoch_end(self, args, state, control, **kwargs):
        if state.epoch is not None and state.epoch >= self.epocas - 1e-6:
            control.should_training_stop = True
        return control


## 6. Argumentos de entrenamiento

El Trainer evalúa por época con AdamW + 10% warmup. En los **Inner Folds** no se guardan checkpoints: los modelos internos se descartan y el F1 y la mejor época se leen de la curva; `ParadaTemprana` detiene el entrenamiento si el F1-Macro no mejora durante `PACIENCIA_EARLY_STOPPING` épocas. En el **refit externo y final no hay early stopping** (no se puede usar el Outer Test ni el test oficial como validación): se entrena con el scheduler de `num_train_epochs` y `DetenerEnEpoca` corta en la mediana de las mejores épocas internas.

In [ ]:
def argumentos_entrenamiento(carpeta, cfg, seed, con_validacion=True, epochs=None):
    params = {
        "output_dir": str(carpeta),
        "learning_rate": cfg["learning_rate"],
        "per_device_train_batch_size": cfg["batch_size"],
        "per_device_eval_batch_size": cfg["batch_size"],
        "num_train_epochs": epochs if epochs is not None else cfg["num_train_epochs"],
        "weight_decay": cfg["weight_decay"],
        "optim": "adamw_torch",
        # Trainer recoge loss por época en state.log_history;
        # PrinterCallback se elimina para no saturar VS Code.
        "logging_strategy": "epoch",
        "save_strategy": "epoch",  # Inner y refit: reinicio desde última época completa.
        "load_best_model_at_end": bool(con_validacion),
        "save_total_limit": 2,  # último + mejor (si son diferentes)
        "save_only_model": False,  # Estado optimizer/scheduler requerido para reanudar.
        "report_to": "none",
        "disable_tqdm": True,
        # Necesario para que EarlyStoppingCallback conserve su contador
        # al reanudar desde un checkpoint.
        "restore_callback_states_from_checkpoint": True,
        "dataloader_num_workers": 0,
        "seed": seed,
        "data_seed": seed,
        "bf16": USE_BF16,
        "fp16": USE_FP16,
    }
    if con_validacion:
        # v4 — Inner Folds sin checkpoints: el modelo interno se descarta y F1/mejor época
        # se leen de la curva. El early stopping lo hace ParadaTemprana.
        params.update({"save_strategy": "no", "load_best_model_at_end": False})
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        params["eval_strategy"] = "epoch" if con_validacion else "no"
    elif "evaluation_strategy" in sig:
        params["evaluation_strategy"] = "epoch" if con_validacion else "no"
    else:
        raise RuntimeError("Esta versión de Transformers no admite evaluación por época.")
    # v5: warmup explícito (transformers 5: `warmup_steps` acepta una fracción en [0, 1)).
    if "warmup_ratio" in sig:
        params["warmup_ratio"] = float(WARMUP_FRACCION)
    else:
        params["warmup_steps"] = float(WARMUP_FRACCION)
    # v5: nunca descartar argumentos en silencio (así se perdió el warmup de la v4).
    descartados = sorted(k for k in params if k not in sig)
    if descartados:
        raise RuntimeError(f"TrainingArguments no acepta {descartados} en esta versión de transformers.")
    return TrainingArguments(**params)


def mejor_epoca(trainer, max_epochs):
    evaluaciones = [x for x in trainer.state.log_history if "eval_f1_macro" in x and "epoch" in x]
    if not evaluaciones:
        raise RuntimeError("No se encontró F1 de validación por época.")
    mejor = max(evaluaciones, key=lambda x: x["eval_f1_macro"])
    return int(np.clip(round(float(mejor["epoch"])), 1, max_epochs))


def predecir(trainer, dataset):
    logits = trainer.predict(dataset).predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    return np.argmax(logits, axis=-1)

## 7. Monitor único y archivos de recuperación

El panel se actualiza sin crear cientos de outputs. Al terminar una etapa (Nested CV, optimización final o test) el panel pasa a **verde con «✅ TERMINADO»**, la barra llega al 100 %, se imprime un mensaje con la hora en la celda y suena un aviso en Windows (`AVISO_SONORO`). Si se detiene por error o interrupción, el panel pasa a **rojo con «❌ DETENIDO»**. Se registran señales en `monitor.log` aproximadamente cada 30 segundos durante los pasos de entrenamiento. Los trials completos se guardan en una base SQLite por Outer Fold y los resultados de cada Outer Fold se escriben inmediatamente en CSV.

Si un entrenamiento se interrumpe, puedes ejecutar otra vez el notebook con **los mismos CSV y la misma configuración**. Se comprueba la configuración guardada para evitar mezclar experimentos distintos.

In [ ]:
class Monitor:
    """Panel único de progreso + monitor.log, con estado activo / terminado / error.

    Clase común a todos los notebooks de modelos (ML, Hard Voting y Transformers).
    """

    def __init__(self, ruta, total, ya_completados=0, unidad="Fine-tunings completados"):
        self.ruta = Path(ruta)
        self.ruta.mkdir(parents=True, exist_ok=True)
        self.log = self.ruta / "monitor.log"
        self.total = int(total)
        self.unidad = unidad
        self.completados = int(ya_completados)
        self.completados_inicio = int(ya_completados)
        self.inicio = time.monotonic()
        self.ultimo_pulso = 0
        self.etapa = "Inicializando"
        self.estado = "activo"  # activo | terminado | error
        self.handle = display(HTML(f"<b>Preparando {NOMBRE_EXPERIMENTO}...</b>"), display_id=True)
        self.mostrar(forzar=True)

    def registrar(self, msg):
        with self.log.open("a", encoding="utf-8") as f:
            f.write(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}\n")

    def mostrar(self, etapa=None, forzar=False):
        if etapa is not None:
            self.etapa = str(etapa)
        ahora = time.monotonic()
        if not forzar and ahora - self.ultimo_pulso < HEARTBEAT_SEGUNDOS:
            return
        self.ultimo_pulso = ahora
        minutos = (ahora - self.inicio) / 60
        hechos_sesion = self.completados - self.completados_inicio
        faltan = max(0, self.total - self.completados)
        eta = (
            f"~{minutos / hechos_sesion * faltan:.0f} min"
            if hechos_sesion >= 2
            else "calculando..."
        )
        marca = datetime.now().strftime("%H:%M:%S")
        color, titulo = {
            "activo": ("#777", f"{NOMBRE_EXPERIMENTO} — seguimiento en vivo"),
            "terminado": ("#2e7d32", f"✅ {NOMBRE_EXPERIMENTO} — TERMINADO"),
            "error": ("#c62828", f"❌ {NOMBRE_EXPERIMENTO} — DETENIDO"),
        }[self.estado]
        grosor = 1 if self.estado == "activo" else 3
        tiempo = (
            f"Tiempo: {minutos:.1f} min | ETA orientativo: {eta}"
            if self.estado == "activo"
            else f"Duración total: {minutos:.1f} min | Finalizó a las {marca}"
        )
        panel = (
            f'<div style="font-family:system-ui;padding:12px;border:{grosor}px solid {color};'
            'border-radius:8px;max-width:800px">'
            f'<b style="color:{color}">{titulo}</b><br>'
            f"Etapa: <b>{html.escape(self.etapa)}</b><br>"
            f"{self.unidad}: {self.completados}/{self.total} "
            f"({100*self.completados/max(1,self.total):.1f}%)<br>"
            f'<progress value="{self.completados}" max="{max(1,self.total)}" '
            'style="width:100%"></progress><br>'
            f"{tiempo}<br>"
            f"Última señal: {marca}<br>Log: {html.escape(str(self.log))}</div>"
        )
        self.handle.update(HTML(panel))

    def iniciar(self, msg):
        self.registrar(msg)
        self.mostrar(etapa=msg, forzar=True)

    def pulso(self):
        if time.monotonic() - self.ultimo_pulso >= HEARTBEAT_SEGUNDOS:
            self.registrar("ACTIVO: " + self.etapa)
            self.mostrar(forzar=True)

    def fit_terminado(self, msg):
        self.completados += 1
        self.registrar(msg)
        self.mostrar(etapa=msg, forzar=True)

    def finalizar(self, msg):
        """Etapa terminada: panel verde al 100 %, mensaje en la celda y aviso sonoro."""
        self.registrar(f"FINALIZADO: {msg} ({self.completados}/{self.total} contados)")
        self.estado = "terminado"
        self.completados = max(self.completados, self.total)
        self.mostrar(etapa=msg, forzar=True)
        print(f"\n✅ [{NOMBRE_EXPERIMENTO}] {msg} — {datetime.now():%Y-%m-%d %H:%M:%S}", flush=True)
        avisar_fin(ok=True)

    def fallar(self, msg):
        """Etapa detenida (error o interrupción): panel rojo, mensaje en la celda y aviso sonoro."""
        self.registrar(f"DETENIDO: {msg}")
        self.estado = "error"
        self.mostrar(etapa=msg, forzar=True)
        print(f"\n❌ [{NOMBRE_EXPERIMENTO}] {msg} — {datetime.now():%Y-%m-%d %H:%M:%S}", flush=True)
        avisar_fin(ok=False)


def avisar_fin(ok=True):
    """Pitido de Windows al terminar (3 veces) o al detenerse (1 vez); en otros sistemas no hace nada."""
    if not AVISO_SONORO:
        return
    try:
        import winsound

        for _ in range(3 if ok else 1):
            winsound.MessageBeep(winsound.MB_ICONASTERISK if ok else winsound.MB_ICONHAND)
            time.sleep(0.5)
    except Exception:
        pass


class PulsoTrainer(TrainerCallback):
    def __init__(self, monitor):
        self.monitor = monitor

    def on_step_end(self, args, state, control, **kwargs):
        self.monitor.pulso()
        return control


def entrenador_silencioso(trainer):
    trainer.remove_callback(PrinterCallback)
    trainer.remove_callback(ProgressCallback)
    return trainer


def csv_atomico(df, ruta):
    ruta = Path(ruta)
    tmp = ruta.with_name(ruta.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, ruta)


def configuracion_actual():
    return {
        "checkpoint": MODEL_NAME,
        "modo": MODO,
        "variantes": VARIANTES,
        "outer": N_OUTER,
        "inner": N_INNER,
        "trials": N_TRIALS,
        "startup_trials": N_STARTUP_TRIALS,
        "semilla": SEED,
        "max_length": MAX_LENGTH,
        "search_space": SEARCH_SPACE,
        "preprocessing_pipeline": PIPELINE_VERSION_ESPERADA,
        "train_final": {
            str(fila.variante): int(fila.train_final)
            for fila in MANIFEST_PREPROCESAMIENTO.itertuples()
        },
        "group_strategy": "original_"
        + "_".join(TEXT_COLS).lower()
        + "_tokenized_connected_components_v1",
        "fixed_training": {
            "optim": "adamw_torch",
            "warmup_fraccion": WARMUP_FRACCION,
            "early_stopping_patience": PACIENCIA_EARLY_STOPPING,
            "early_stopping": "parada_temprana_f1_sin_checkpoints_v4",
            "refit_epoch_strategy": "median_inner_best_epoch_rounded_same_scheduler_v4",
            "loss": "ce_label_smoothing_train_only_v4",
            "optimizer": "adamw_layerwise_lr_decay_v4",
            "regularizacion_v5": "dropout_interno_y_congelar_frac_en_busqueda",
            "metricas_v5": "nll_y_ece_sin_pesos",
            "curvas": "train_eval_loss_epoch_csv_best_outer_png_v1",
            "checkpoint": "refit_and_test_only_v4",
        },
    }


def verificar_configuracion():
    ruta = RUN_DIR / "config_ejecucion.json"
    actual = configuracion_actual()
    if ruta.exists():
        anterior = json.loads(ruta.read_text(encoding="utf-8"))
        if anterior != actual:
            raise ValueError(
                "Esta carpeta tiene una corrida con datos/configuración diferentes. "
                "No mezcles resultados; cambia MODO/carpeta o archiva el experimento."
            )
    else:
        tmp = ruta.with_suffix(".tmp")
        tmp.write_text(json.dumps(actual, indent=2, ensure_ascii=False), encoding="utf-8")
        os.replace(tmp, ruta)


def ruta_estudio(variante, nombre):
    return RUN_DIR / f"robertuito_{variante}_{nombre}.db"


def cargar_study(variante, nombre):
    ruta = ruta_estudio(variante, nombre)
    if not ruta.exists():
        return None
    return optuna.load_study(
        study_name=f"RoBERTuito_{variante}_{nombre}",
        storage=f"sqlite:///{ruta.resolve().as_posix()}",
    )


def contar_completados():
    n = 0
    for variante in VARIANTES:
        ruta_csv = RUN_DIR / f"robertuito_detalle_{variante}.csv"
        completados = set()
        if ruta_csv.exists():
            completados = set(pd.read_csv(ruta_csv)["outer_fold"].astype(int))
        for outer in range(1, N_OUTER + 1):
            if outer in completados:
                n += N_TRIALS * N_INNER + 1
            else:
                st = cargar_study(variante, f"outer{outer}")
                if st is not None:
                    n += (
                        min(
                            N_TRIALS,
                            sum(t.state == optuna.trial.TrialState.COMPLETE for t in st.trials),
                        )
                        * N_INNER
                    )
    return n

### 7.1 Checkpoints por época y curvas de pérdida

Las carpetas de cada entrenamiento se identifican por **variante, Outer Fold, configuración exacta e Inner Fold**, no por número de trial: si Optuna debe reencolar un trial interrumpido, se pueden recuperar sus Inner Folds sin perderlos. El estado se guarda al finalizar cada época; la validación de cada entrenamiento usa solo su Inner Fold, nunca el test oficial.

Los CSV contienen `epoch`, `train_loss`, `val_loss` y `val_f1_macro`. Se generan gráficos por Outer Fold del **mejor trial según Inner CV**, con las tres particiones internas; al acabar cada variante se pueden consultar en `graphs/<variante>/`. Las épocas que no alcanzan todos los folds por early stopping se indican con `n_folds` en el CSV agregado.

In [ ]:
# ==============================================================
# CHECKPOINTS, CURVAS Y REANUDACIÓN — ESQUEMA COMÚN A LOS TRES TRANSFORMERS
# ==============================================================
def config_id(cfg):
    """ID compacto y determinista para evitar rutas largas en Windows."""
    prep = {"PYSENTIMIENTO": "P"}[cfg["preprocessing"]]
    lr = int(round(float(cfg["learning_rate"]) * 1e5))  # 5e-5 -> 5
    bs = int(cfg["batch_size"])
    ep = int(cfg["num_train_epochs"])
    wd = int(round(float(cfg["weight_decay"]) * 100))  # .05 -> 5
    loss = {"standard": "s", "balanced": "b"}[cfg["loss_mode"]]
    drop = int(round(float(cfg["classifier_dropout"]) * 10))  # .2 -> 2, .3 -> 3
    ls = int(round(float(cfg.get("label_smoothing", 0.0)) * 10))  # .1 -> 1
    lrd = int(round(float(cfg.get("lr_decay", 1.0)) * 10))  # .9 -> 9
    di = int(round(float(cfg.get("dropout_interno", 0.1)) * 10))  # v5: .2 -> 2
    cf = int(round(float(cfg.get("congelar_frac", 0.0)) * 100))  # v5: .25 -> 25
    mk = int(round(float(cfg.get("mascara_tokens", 0.0)) * 100))  # v6: .15 -> 15
    return f"{prep}_l{lr}_b{bs}_e{ep}_w{wd}_{loss}_d{drop}_s{ls}_r{lrd}_i{di}_c{cf}_m{mk}"


def rutas_inner_fit(variante, etiqueta, cfg, inner_fold):
    """
    Estructura común a los tres Transformers:
      ckpt/mx/outer1/S_l5_b16_e5_w5_s_d3/inner1/
      state/mx_outer1_S_l5_b16_e5_w5_s_d3_inner1_complete.json
      curves/mx_outer1_S_l5_b16_e5_w5_s_d3_inner1.csv
    """
    ident = config_id(cfg)
    base = f"{variante}_{etiqueta}_{ident}_inner{inner_fold}"
    return {
        "config_id": ident,
        "state": FIT_STATE_DIR / f"{base}_complete.json",
        "curve": CURVE_DIR / f"{base}.csv",
        "ckpt": CHECKPOINT_DIR / variante / etiqueta / ident / f"inner{inner_fold}",
    }


def cargar_estado_inner_si_completo(rutas):
    if not rutas["state"].exists():
        return None
    return json.loads(rutas["state"].read_text(encoding="utf-8"))


def json_atomico(obj, ruta):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    temporal = ruta.with_name(ruta.name + ".tmp")
    temporal.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(temporal, ruta)


def extraer_curva(log_history):
    """Empareja train/validation por época, sin tocar Outer Test."""
    filas = {}
    for entry in log_history:
        if "epoch" not in entry:
            continue

        epoch = round(float(entry["epoch"]), 6)
        if epoch <= 0:
            continue

        fila = filas.setdefault(
            epoch,
            {"epoch": epoch, "train_loss": np.nan, "val_loss": np.nan, "val_f1_macro": np.nan},
        )

        if "loss" in entry:
            fila["train_loss"] = float(entry["loss"])
        if "eval_loss" in entry:
            fila["val_loss"] = float(entry["eval_loss"])
        if "eval_f1_macro" in entry:
            fila["val_f1_macro"] = float(entry["eval_f1_macro"])
        if "eval_nll" in entry:  # v5: NLL sin pesos de clase
            fila["val_nll"] = float(entry["eval_nll"])

    if not filas:
        return pd.DataFrame(columns=["epoch", "train_loss", "val_loss", "val_f1_macro"])

    return pd.DataFrame(list(filas.values())).sort_values("epoch").reset_index(drop=True)


def obtener_mejor_epoca_desde_curva(curva, max_epochs):
    if curva.empty or "val_f1_macro" not in curva or curva["val_f1_macro"].dropna().empty:
        return int(max_epochs)

    fila = curva.loc[curva["val_f1_macro"].idxmax()]
    return max(1, min(int(max_epochs), int(round(float(fila["epoch"])))))


def limpiar_checkpoint_dir(carpeta):
    if CLEAN_COMPLETED_CHECKPOINTS:
        shutil.rmtree(carpeta, ignore_errors=True)


def ejecutar_inner_fit_reanudable(
    cache, y, idx_train, idx_val, cfg, variante, etiqueta, inner_fold, seed_modelo, monitor=None
):
    rutas = rutas_inner_fit(variante, etiqueta, cfg, inner_fold)

    previo = cargar_estado_inner_si_completo(rutas)
    if previo is not None:
        if previo.get("config_id") != rutas["config_id"]:
            raise ValueError("Estado Inner almacenado no coincide con la configuración.")

        if monitor:
            monitor.fit_terminado(
                f"REUTILIZADO {variante.upper()} {etiqueta} "
                f"cfg={rutas['config_id']} inner={inner_fold}: "
                f"F1={float(previo['f1_inner']):.4f}"
            )

        return (float(previo["f1_inner"]), int(previo["best_epoch"]), rutas)

    rutas["ckpt"].mkdir(parents=True, exist_ok=True)

    dataset_completo = cache[cfg["preprocessing"]]
    train_ds = Subset(dataset_completo, list(map(int, idx_train)))
    val_ds = Subset(dataset_completo, list(map(int, idx_val)))

    if monitor:
        monitor.iniciar(
            f"{variante.upper()} {etiqueta} | "
            f"cfg={rutas['config_id']} | "
            f"Inner {inner_fold}/{N_INNER}"
        )

    model = crear_modelo(seed_modelo, cfg["classifier_dropout"], cfg)

    class_weights = pesos_balanceados(y[idx_train]) if cfg["loss_mode"] == "balanced" else None

    trainer = None
    try:
        args = argumentos_entrenamiento(rutas["ckpt"], cfg, seed_modelo, con_validacion=True)

        callbacks = [ParadaTemprana(PACIENCIA_EARLY_STOPPING)]

        if monitor:
            callbacks.append(PulsoTrainer(monitor))

        trainer = entrenador_silencioso(
            ClasificadorTrainer(
                model=model,
                args=args,
                train_dataset=train_ds,
                eval_dataset=val_ds,
                data_collator=data_collator,
                compute_metrics=metricas,
                callbacks=callbacks,
                class_weights=class_weights,
                label_smoothing=cfg["label_smoothing"],
                lr_decay=cfg["lr_decay"],
                mascara_tokens=cfg.get("mascara_tokens", 0.0),
            )
        )

        ultimo = get_last_checkpoint(str(rutas["ckpt"]))

        if ultimo and monitor:
            monitor.registrar(f"Reanudando fit desde {ultimo}")

        trainer.train(resume_from_checkpoint=(ultimo if ultimo else None))

        # Congelar la historia de entrenamiento ANTES de cualquier evaluación extra.
        curva = extraer_curva(trainer.state.log_history)

        if curva.empty or curva["val_loss"].isna().all() or curva["train_loss"].isna().all():
            raise RuntimeError("Faltan las curvas train/validation loss.")

        csv_atomico(curva, rutas["curve"])

        best_epoch = obtener_mejor_epoca_desde_curva(curva, cfg["num_train_epochs"])

        # v4: sin checkpoints; el F1 interno es el mejor F1 de validación de la curva.
        f1 = float(curva["val_f1_macro"].max())

        json_atomico(
            {
                "status": "complete",
                "config_id": rutas["config_id"],
                "config": cfg,
                "f1_inner": f1,
                "best_epoch": int(best_epoch),
                "inner_fold": int(inner_fold),
                "seed": int(seed_modelo),
                "checkpoint_used_to_resume": ultimo,
            },
            rutas["state"],
        )

    finally:
        if trainer is not None:
            del trainer
        del model
        liberar_memoria()

    # Al acabar correctamente, borrar únicamente los
    # checkpoints pesados. El JSON y la curva quedan fuera de esa carpeta.
    limpiar_checkpoint_dir(rutas["ckpt"])

    if monitor:
        monitor.fit_terminado(
            f"{variante.upper()} {etiqueta} "
            f"cfg={rutas['config_id']} "
            f"Inner {inner_fold}: F1={f1:.4f}"
        )

    return f1, best_epoch, rutas


def graficar_mejor_trial(variante, outer_fold, cfg, n_inner, mostrar=False):
    """Usa solo curvas de Inner Validation del mejor trial."""
    etiqueta = f"outer{outer_fold}"
    curves = []

    for fold in range(1, n_inner + 1):
        ruta = rutas_inner_fit(variante, etiqueta, cfg, fold)["curve"]

        if not ruta.exists():
            raise FileNotFoundError(f"Falta curva del mejor trial: {ruta}")

        d = pd.read_csv(ruta)
        d["inner_fold"] = fold
        curves.append(d)

    todos = pd.concat(curves, ignore_index=True)

    resumen = todos.groupby("epoch", as_index=False).agg(
        train_loss=("train_loss", "mean"),
        val_loss=("val_loss", "mean"),
        train_std=("train_loss", "std"),
        val_std=("val_loss", "std"),
        val_f1_macro=("val_f1_macro", "mean"),
        n_folds=("inner_fold", "nunique"),
    )

    salida = GRAPH_DIR / variante
    salida.mkdir(parents=True, exist_ok=True)

    csv_atomico(todos, salida / f"outer{outer_fold}_inner_folds.csv")
    csv_atomico(resumen, salida / f"outer{outer_fold}_media_por_epoca.csv")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(resumen["epoch"], resumen["train_loss"], marker="o", label="Train loss")
    ax.plot(resumen["epoch"], resumen["val_loss"], marker="o", label="Validation loss")
    ax.set(
        xlabel="Época",
        ylabel="Loss",
        title=(f"RoBERTuito {variante.upper()} — Outer {outer_fold} | mejor trial interno"),
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    loss_png = salida / f"outer{outer_fold}_loss.png"
    fig.savefig(loss_png, dpi=160, bbox_inches="tight")
    if mostrar:
        display(fig)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(resumen["epoch"], resumen["val_f1_macro"], marker="o")
    ax.set(
        xlabel="Época",
        ylabel="F1-Macro validación",
        title=(f"RoBERTuito {variante.upper()} — Outer {outer_fold} | mejor trial interno"),
    )
    ax.grid(alpha=0.25)
    fig.tight_layout()

    f1_png = salida / f"outer{outer_fold}_f1.png"
    fig.savefig(f1_png, dpi=160, bbox_inches="tight")
    if mostrar:
        display(fig)
    plt.close(fig)

    return {
        "csv_inner": str(salida / f"outer{outer_fold}_inner_folds.csv"),
        "csv_media": str(salida / f"outer{outer_fold}_media_por_epoca.csv"),
        "loss_png": str(loss_png),
        "f1_png": str(f1_png),
    }

In [ ]:
def cargar_test(variante):
    ruta = DATA_DIR / f"test_transformers_{variante}.csv"
    if not ruta.exists():
        raise FileNotFoundError(ruta)
    df = pd.read_csv(ruta)
    req = [ID_COL, ORIGINAL_COL, LABEL_COL, *TEXT_COLS.values()]
    if any(col not in df.columns for col in req):
        raise ValueError(f"Test {variante} no contiene todas las columnas esperadas.")
    df[ID_COL] = df[ID_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    df[ORIGINAL_COL] = df[ORIGINAL_COL].fillna("").astype(str)
    for col in TEXT_COLS.values():
        df[col] = df[col].fillna("").astype(str)
    if len(df) != 600:
        raise ValueError(f"Se esperan 600 ejemplos del test oficial {variante}, hay {len(df)}.")
    return df.reset_index(drop=True)


def detectar_coincidencias_test(train_df, test_df, prep, variante):
    """Marca los textos del test que coinciden con alguno del train.

    Solo alimenta el análisis de sensibilidad del test oficial: no elimina
    registros ni interviene en ninguna decisión del modelo.
    Coincidencia = mismo MESSAGE original (espacios colapsados) o misma
    secuencia de tokens de entrada del modelo (truncada a MAX_LENGTH).
    """
    col = TEXT_COLS[prep]

    def claves_tokens(textos):
        ids = tokenizer(
            [str(t) for t in textos], truncation=True, max_length=MAX_LENGTH, padding=False
        )["input_ids"]
        return [tuple(x) for x in ids]

    originales_train = set(train_df[ORIGINAL_COL].map(colapsar_espacios))
    tokens_train = set(claves_tokens(train_df[col]))
    coincide_original = (
        test_df[ORIGINAL_COL].map(colapsar_espacios).isin(originales_train).to_numpy()
    )
    coincide_tokens = np.array([k in tokens_train for k in claves_tokens(test_df[col])], dtype=bool)
    mascara = coincide_original | coincide_tokens

    detalle = test_df.loc[mascara, [ID_COL, ORIGINAL_COL, col]].copy()
    detalle["coincide_original"] = coincide_original[mascara]
    detalle["coincide_tokens"] = coincide_tokens[mascara]
    csv_atomico(detalle, AUDIT_DIR / f"coincidencias_test_train_{variante}_{prep}.csv")
    return mascara


def predecir_probabilidades(trainer, dataset):
    """Probabilidades softmax por clase (columna 1 = irónico)."""
    logits = trainer.predict(dataset).predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(logits)
    return probs / probs.sum(axis=1, keepdims=True)


def evaluar_semilla_final(train_df, test_df, cfg, seed, variante, mascara_coincide, monitor=None):
    prep = cfg["preprocessing"]
    ds_train = DatasetTokenizado(train_df[TEXT_COLS[prep]], train_df[LABEL_COL])
    ds_test = DatasetTokenizado(test_df[TEXT_COLS[prep]], test_df[LABEL_COL])
    y_train = train_df[LABEL_COL].to_numpy(dtype=int)
    y_test = test_df[LABEL_COL].to_numpy(dtype=int)
    sin_coincidencias = ~np.asarray(mascara_coincide, dtype=bool)
    carpeta = CHECKPOINT_DIR / variante / "test_final" / f"seed{seed}"
    carpeta.mkdir(parents=True, exist_ok=True)
    resultado_file = FIT_STATE_DIR / f"{variante}_test_seed{seed}_complete.json"
    cfg_id = config_id(cfg)
    claves = (
        "seed",
        "f1_macro",
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro_sin_coincidencias",
        "n_coincidencias",
        "nll",
        "ece",
        "nll_calibrada",
        "ece_calibrada",
        "temperatura",
    )
    pred_file = FIT_STATE_DIR / f"{variante}_test_seed{seed}_predicciones.csv"
    if resultado_file.exists():
        if not pred_file.exists():
            raise FileNotFoundError(
                f"Falta {pred_file}; borra {resultado_file} y repite esa semilla."
            )
        previo = json.loads(resultado_file.read_text(encoding="utf-8"))
        if previo.get("config_id") != cfg_id:
            raise ValueError("Configuración final distinta a la guardada para la semilla")
        return {k: previo.get(k) for k in claves}
    if monitor:
        monitor.iniciar(f"{variante.upper()} | Test oficial | semilla {seed}")
    cw = pesos_balanceados(y_train) if cfg["loss_mode"] == "balanced" else None
    model = crear_modelo(seed, cfg["classifier_dropout"], cfg)
    trainer = None
    try:
        args = argumentos_entrenamiento(
            carpeta, cfg, seed, con_validacion=False, epochs=cfg["num_train_epochs"]
        )
        trainer = entrenador_silencioso(
            ClasificadorTrainer(
                model=model,
                args=args,
                train_dataset=ds_train,
                data_collator=data_collator,
                class_weights=cw,
                label_smoothing=cfg["label_smoothing"],
                lr_decay=cfg["lr_decay"],
                mascara_tokens=cfg.get("mascara_tokens", 0.0),
                callbacks=[DetenerEnEpoca(cfg["refit_epochs"])]
                + ([PulsoTrainer(monitor)] if monitor else []),
            )
        )
        ultimo = get_last_checkpoint(str(carpeta))
        trainer.train(resume_from_checkpoint=ultimo if ultimo else None)
        probs = predecir_probabilidades(trainer, ds_test)
        y_pred = probs.argmax(axis=1)
        # v5: temperatura ajustada SOLO con predicciones out-of-fold del train (sección 16-bis).
        ruta_T = RESULTS_DIR / f"{NOMBRE_EXPERIMENTO.lower()}_temperatura.json"
        T = json.loads(ruta_T.read_text(encoding="utf-8"))[variante]["T"] if ruta_T.exists() else 1.0
        probs_cal = softmax_np(np.log(np.clip(probs, 1e-12, 1.0)) / T)
        # Predicciones y probabilidades por texto: base de la ROC, la curva PR
        # y las matrices de confusión del notebook de comparación final.
        csv_atomico(
            pd.DataFrame(
                {
                    "variante": variante,
                    "seed": int(seed),
                    "ID": test_df[ID_COL].astype(str).to_numpy(),
                    "y_true": y_test,
                    "y_pred": y_pred,
                    "prob_ironia": probs[:, 1],
                    "prob_ironia_calibrada": probs_cal[:, 1],
                    "coincide_train": ~sin_coincidencias,
                }
            ),
            pred_file,
        )
        curva = extraer_curva(trainer.state.log_history)
        if not curva.empty:
            csv_atomico(curva, CURVE_DIR / f"{variante}_test_seed{seed}.csv")
        resultado = {
            "seed": int(seed),
            "f1_macro": float(f1_score(y_test, y_pred, average="macro")),
            "accuracy": float(accuracy_score(y_test, y_pred)),
            "precision_macro": float(
                precision_score(y_test, y_pred, average="macro", zero_division=0)
            ),
            "recall_macro": float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
            # Análisis de sensibilidad: mismo modelo, sin los textos que coinciden con el train.
            "f1_macro_sin_coincidencias": float(
                f1_score(y_test[sin_coincidencias], y_pred[sin_coincidencias], average="macro")
            ),
            "n_coincidencias": int((~sin_coincidencias).sum()),
            "nll": nll_binaria(probs, y_test),
            "ece": ece_binaria(probs, y_test),
            "nll_calibrada": nll_binaria(probs_cal, y_test),
            "ece_calibrada": ece_binaria(probs_cal, y_test),
            "temperatura": float(T),
        }
        json_atomico({"config_id": cfg_id, **resultado}, resultado_file)
        limpiar_checkpoint_dir(carpeta)
    finally:
        if trainer is not None:
            del trainer
        del model
        liberar_memoria()
    if monitor:
        monitor.fit_terminado(
            f"{variante.upper()} test semilla {seed}: F1={resultado['f1_macro']:.4f}"
        )
    return resultado

## 8. Configuración final de la v6 (no se repite la búsqueda)

La v7 **no vuelve a ejecutar** la Nested CV, Optuna ni el test de 5 semillas: esos resultados ya están en la carpeta de la v6 y no cambian. Aquí se leen:

- `<modelo>_config_final.json`: hiperparámetros finales por variante (elegidos solo con el train) y `refit_epochs`;
- `<modelo>_temperatura.json`: temperatura ajustada con predicciones fuera de fold del train;
- `<modelo>_test_probabilidades.csv`: predicciones de la v6 por semilla (se usa la semilla 42 para verificar la reproducción).

Se comprueba que los valores fijos de la configuración coinciden con `SEARCH_SPACE`, para no mezclar versiones.

In [ ]:
PREFIJO = NOMBRE_EXPERIMENTO.lower()
RUTA_CFG_V6 = RESULTS_DIR_V6 / f"{PREFIJO}_config_final.json"
RUTA_T_V6 = RESULTS_DIR_V6 / f"{PREFIJO}_temperatura.json"
RUTA_PRED_V6 = RESULTS_DIR_V6 / f"{PREFIJO}_test_probabilidades.csv"
for ruta in (RUTA_CFG_V6, RUTA_T_V6, RUTA_PRED_V6):
    if not ruta.exists():
        raise FileNotFoundError(
            f"Falta {ruta}. Copia a esta máquina la carpeta de resultados v6 de {NOMBRE_EXPERIMENTO}."
        )

configs_finales = json.loads(RUTA_CFG_V6.read_text(encoding="utf-8"))
TEMPERATURAS = {v: float(d["T"]) for v, d in json.loads(RUTA_T_V6.read_text(encoding="utf-8")).items()}
PRED_V6 = pd.read_csv(RUTA_PRED_V6, dtype={"ID": str})

for variante in VARIANTES:
    cfg = configs_finales[variante]
    faltan = sorted((SEARCH_SPACE_REQUERIDO | {"refit_epochs"}) - set(cfg))
    if faltan:
        raise KeyError(f"{variante}: a la configuración v6 le faltan {faltan}")
    for clave, opciones in SEARCH_SPACE.items():
        if len(opciones) == 1 and cfg[clave] != opciones[0]:
            raise ValueError(f"{variante}: {clave}={cfg[clave]} en la v6 pero {opciones[0]} en SEARCH_SPACE")

columnas_cfg = ["weight_decay", "classifier_dropout", "dropout_interno", "mascara_tokens",
                "label_smoothing", "lr_decay", "learning_rate", "batch_size", "refit_epochs"]
tabla_cfg = pd.DataFrame(configs_finales).T.loc[VARIANTES, columnas_cfg]
tabla_cfg["temperatura"] = [TEMPERATURAS[v] for v in VARIANTES]
print(f"[V7][{NOMBRE_EXPERIMENTO}] Configuración final leída de {RESULTS_DIR_V6}")
display(tabla_cfg)

## 9. Entrenamiento y exportación del modelo final

Para cada variante se entrena **un** modelo con todo el train oficial, la configuración final de la v6 y la semilla `SEMILLA_EXPORTACION = 42`. Es **exactamente el mismo procedimiento** que la semilla 42 del test de la v6 (`evaluar_semilla_final`): mismo `crear_modelo`, mismo `ClasificadorTrainer` (pesos de clase, label smoothing, LR decay por capas, dropout interno y enmascarado si corresponde), mismo scheduler de `num_train_epochs` y corte en `refit_epochs` con `DetenerEnEpoca`.

**Diferencia con la v6:** al terminar, el modelo **se guarda** (la v6 borraba los checkpoints para ahorrar disco). Se exporta en formato Hugging Face en `modelo_final/<variante>/`:

| Archivo | Contenido |
|---|---|
| `model.safetensors` | pesos del modelo ajustado |
| `config.json` | arquitectura, dropout y etiquetas (`0 = NO_IRONICO`, `1 = IRONICO`) |
| `tokenizer.json`, `tokenizer_config.json`, … | tokenizador usado en el entrenamiento |
| `metadatos_v7.json` | configuración, semilla, épocas, temperatura, versiones de los paquetes y huella (SHA-256) del CSV de train |
| `curva_entrenamiento.csv` | pérdida de train por época |

Si el modelo de una variante ya está exportado con la misma configuración y semilla, no se vuelve a entrenar. Ocupa ≈ 0.3–0.5 GB por variante.

In [ ]:
MODELOS_DIR = RESULTS_DIR / "modelo_final"
MODELOS_DIR.mkdir(parents=True, exist_ok=True)
ETIQUETAS = {0: "NO_IRONICO", 1: "IRONICO"}


def sha256_archivo(ruta):
    import hashlib

    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


def modelo_exportado_valido(variante, cfg):
    meta_path = MODELOS_DIR / variante / "metadatos_v7.json"
    if not (meta_path.exists() and (MODELOS_DIR / variante / "config.json").exists()):
        return False
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    return meta.get("config_id") == config_id(cfg) and meta.get("seed") == SEMILLA_EXPORTACION


def entrenar_y_exportar(variante, monitor=None):
    cfg = configs_finales[variante]
    destino = MODELOS_DIR / variante
    if modelo_exportado_valido(variante, cfg):
        if monitor:
            monitor.fit_terminado(f"{variante.upper()}: modelo ya exportado (se reutiliza)")
        return destino
    if monitor:
        monitor.iniciar(f"{variante.upper()} | Entrenamiento del modelo final (semilla {SEMILLA_EXPORTACION})")
    df_train = cargar_train(variante)
    prep = cfg["preprocessing"]
    ds_train = DatasetTokenizado(df_train[TEXT_COLS[prep]], df_train[LABEL_COL])
    y_train = df_train[LABEL_COL].to_numpy(dtype=int)
    cw = pesos_balanceados(y_train) if cfg["loss_mode"] == "balanced" else None
    carpeta = CHECKPOINT_DIR / variante / "exportacion" / f"seed{SEMILLA_EXPORTACION}"
    carpeta.mkdir(parents=True, exist_ok=True)
    model = crear_modelo(SEMILLA_EXPORTACION, cfg["classifier_dropout"], cfg)
    trainer = None
    try:
        args = argumentos_entrenamiento(
            carpeta, cfg, SEMILLA_EXPORTACION, con_validacion=False, epochs=cfg["num_train_epochs"]
        )
        trainer = entrenador_silencioso(
            ClasificadorTrainer(
                model=model,
                args=args,
                train_dataset=ds_train,
                data_collator=data_collator,
                class_weights=cw,
                label_smoothing=cfg["label_smoothing"],
                lr_decay=cfg["lr_decay"],
                mascara_tokens=cfg.get("mascara_tokens", 0.0),
                callbacks=[DetenerEnEpoca(cfg["refit_epochs"])]
                + ([PulsoTrainer(monitor)] if monitor else []),
            )
        )
        ultimo = get_last_checkpoint(str(carpeta))
        trainer.train(resume_from_checkpoint=ultimo if ultimo else None)
        trainer.model.config.id2label = dict(ETIQUETAS)
        trainer.model.config.label2id = {v: k for k, v in ETIQUETAS.items()}
        destino.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(destino))
        tokenizer.save_pretrained(str(destino))
        curva = extraer_curva(trainer.state.log_history)
        csv_atomico(curva, destino / "curva_entrenamiento.csv")
        json_atomico(
            {
                "modelo_base": MODEL_NAME,
                "version": "v7",
                "variante": variante,
                "seed": SEMILLA_EXPORTACION,
                "config_id": config_id(cfg),
                "configuracion": cfg,
                "epocas_entrenadas": float(trainer.state.epoch or 0),
                "n_train": int(len(df_train)),
                "csv_train": str(validar_archivo_train_con_manifest(variante)[0]),
                "sha256_csv_train": sha256_archivo(validar_archivo_train_con_manifest(variante)[0]),
                "columna_texto": TEXT_COLS[prep],
                "max_length": MAX_LENGTH,
                "temperatura": TEMPERATURAS[variante],
                "etiquetas": {str(k): v for k, v in ETIQUETAS.items()},
                "versiones": VERSIONES,
                "fecha": datetime.now().isoformat(timespec="seconds"),
            },
            destino / "metadatos_v7.json",
        )
    finally:
        if trainer is not None:
            del trainer
        del model
        liberar_memoria()
    limpiar_checkpoint_dir(carpeta)
    if monitor:
        monitor.fit_terminado(f"{variante.upper()}: modelo final exportado en {destino}")
    return destino


monitor_export = Monitor(
    RUN_DIR,
    total=len(VARIANTES),
    ya_completados=sum(modelo_exportado_valido(v, configs_finales[v]) for v in VARIANTES),
    unidad="Modelos finales exportados",
)
try:
    for variante in VARIANTES:
        entrenar_y_exportar(variante, monitor=monitor_export)
except KeyboardInterrupt:
    monitor_export.fallar("Exportación interrumpida; al reejecutar se reanuda desde la última época")
    raise
except Exception as exc:
    monitor_export.registrar(f"ERROR exportación {type(exc).__name__}: {exc}")
    monitor_export.fallar(f"Exportación con error {type(exc).__name__}; consultar monitor.log")
    raise
monitor_export.finalizar("Modelos finales entrenados y exportados")

(MODELOS_DIR / "LEEME.md").write_text(
    f"""# {NOMBRE_EXPERIMENTO} — modelos finales (v7)

Un modelo por variante (`mx/`, `es/`, `cu/`), entrenado con todo el train oficial de IroSvA 2019,
la configuración final de la v6 y la semilla {SEMILLA_EXPORTACION}. Etiquetas: 0 = NO_IRONICO, 1 = IRONICO.

## Cómo cargarlo

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

ruta = "modelo_final/mx"
tokenizer = AutoTokenizer.from_pretrained(ruta)
modelo = AutoModelForSequenceClassification.from_pretrained(ruta).eval()
# El texto debe pasar por el mismo preprocesamiento que el train
# (columna `{TEXT_COLS[configs_finales[VARIANTES[0]]['preprocessing']]}`, ver 01_preprocesamiento_dl_transformers.ipynb).
enc = tokenizer(["texto preprocesado"], truncation=True, max_length={MAX_LENGTH}, return_tensors="pt")
with torch.no_grad():
    prob_ironia = torch.softmax(modelo(**enc).logits, dim=-1)[:, 1]
```

Para probabilidades calibradas, dividir los logits por la temperatura de `metadatos_v7.json` antes del softmax.
""",
    encoding="utf-8",
)
print("Modelos exportados en:", MODELOS_DIR.resolve())

## 10. Verificación de la exportación (reproducción de la semilla 42 de la v6)

Se **carga cada modelo desde el disco** (así se comprueba que la exportación funciona) y se predice el test oficial. Las predicciones se comparan con las de la **semilla 42 de la v6**, que se entrenó con el mismo procedimiento.

- Si coinciden (acuerdo ≈ 100 %), el modelo exportado es el mismo de la v6.
- Diferencias pequeñas son normales: la GPU no es 100 % determinista.

**Esto no es una nueva evaluación del test:** no se elige ni se cambia nada; el F1 se muestra solo para comparar con la v6. Las predicciones de esta sección definen los aciertos y errores que se explican con SHAP.

In [ ]:
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"


def predecir_desde_disco(variante, textos, lote=64):
    ruta = MODELOS_DIR / variante
    tok = AutoTokenizer.from_pretrained(str(ruta))
    mod = AutoModelForSequenceClassification.from_pretrained(str(ruta)).to(DISPOSITIVO).eval()
    salidas = []
    try:
        for i in range(0, len(textos), lote):
            enc = tok(
                [str(t) for t in textos[i : i + lote]],
                truncation=True,
                max_length=MAX_LENGTH,
                padding=True,
                return_tensors="pt",
            ).to(DISPOSITIVO)
            with torch.no_grad():
                salidas.append(mod(**enc).logits.float().cpu().numpy())
    finally:
        del mod
        liberar_memoria()
    return np.vstack(salidas)


def tipo_resultado(y, p):
    return np.select([(y == 1) & (p == 1), (y == 0) & (p == 0), (y == 0) & (p == 1)], ["VP", "VN", "FP"], "FN")


PRED_V7, filas_verif = {}, []
for variante in VARIANTES:
    cfg = configs_finales[variante]
    df_test = cargar_test(variante)
    textos = df_test[TEXT_COLS[cfg["preprocessing"]]].tolist()
    logits = predecir_desde_disco(variante, textos)
    probs = softmax_np(logits)
    probs_cal = softmax_np(logits / TEMPERATURAS[variante])
    y = df_test[LABEL_COL].to_numpy(dtype=int)
    pred = probs.argmax(axis=1)
    d = pd.DataFrame(
        {
            "variante": variante,
            "ID": df_test[ID_COL].astype(str).to_numpy(),
            "texto": textos,
            "mensaje_original": df_test[ORIGINAL_COL].to_numpy(),
            "y_true": y,
            "y_pred": pred,
            "prob_ironia": probs[:, 1],
            "prob_ironia_calibrada": probs_cal[:, 1],
            "tipo": tipo_resultado(y, pred),
        }
    )
    PRED_V7[variante] = d
    v6 = PRED_V6[(PRED_V6["variante"] == variante) & (PRED_V6["seed"] == SEMILLA_EXPORTACION)].set_index("ID")
    v6 = v6.loc[d["ID"]]
    filas_verif.append(
        {
            "variante": variante,
            "acuerdo_prediccion": float((v6["y_pred"].to_numpy() == pred).mean()),
            "dif_prob_media": float(np.abs(v6["prob_ironia"].to_numpy() - probs[:, 1]).mean()),
            "dif_prob_max": float(np.abs(v6["prob_ironia"].to_numpy() - probs[:, 1]).max()),
            "f1_v6_seed42": float(f1_score(v6["y_true"], v6["y_pred"], average="macro")),
            "f1_v7_exportado": float(f1_score(y, pred, average="macro")),
        }
    )

VERIFICACION = pd.DataFrame(filas_verif)
VERIFICACION["dif_f1"] = VERIFICACION["f1_v7_exportado"] - VERIFICACION["f1_v6_seed42"]
csv_atomico(VERIFICACION, RESULTS_DIR / f"{PREFIJO}_v7_verificacion_exportacion.csv")
csv_atomico(pd.concat(PRED_V7.values(), ignore_index=True), RESULTS_DIR / f"{PREFIJO}_v7_test_predicciones.csv")
display(VERIFICACION.round(4))
for fila in VERIFICACION.itertuples():
    estado = "OK" if fila.acuerdo_prediccion >= 0.95 else "REVISAR (diferencia mayor a la esperable)"
    print(f"[{fila.variante.upper()}] acuerdo con la v6 = {100 * fila.acuerdo_prediccion:.1f} % → {estado}")

## 11. Interpretabilidad con SHAP

**SHAP** (Lundberg y Lee, 2017) reparte la predicción de cada tuit entre sus tokens: tapa combinaciones de tokens, mide cuánto cambia la probabilidad de **IRONICO** y asigna a cada token su contribución (valores de Shapley). Un valor **positivo** empuja la predicción hacia *irónico*; uno **negativo**, hacia *no irónico*. Se usa el explicador de partición de `shap` para textos con el tokenizador del modelo.

Se eligió SHAP y no los pesos de atención porque la atención no siempre refleja la contribución real de cada token a la predicción (limitación que discuten Ortega-Bueno et al., 2022, en IroSvA).

**Muestra:** por variante, hasta `N_SHAP_POR_TIPO` tuits de cada tipo de resultado del modelo exportado (verdaderos positivos VP, verdaderos negativos VN, falsos positivos FP y falsos negativos FN), elegidos al azar con la semilla 42. Se explica la probabilidad **sin** calibrar (la temperatura no cambia el orden de las contribuciones).

**Salidas:**

- `<modelo>_v7_shap_tokens.csv`: contribución de cada token en cada tuit explicado;
- `<modelo>_v7_shap_global.csv`: por variante y token (en ≥ `MIN_TEXTOS_TOKEN` tuits), contribución media y media absoluta;
- `<modelo>_v7_shap_global.png/.pdf`: tokens que más empujan hacia irónico y hacia no irónico, por variante;
- `<modelo>_v7_shap_ejemplos.png/.pdf` y `shap_html/`: ejemplos locales (VP, FP y FN más seguros) por variante.

Requiere el paquete `shap` (`py -m uv add shap`).

In [ ]:
try:
    import shap
except ImportError as exc:
    raise ImportError("Falta el paquete shap. Instálalo con:  py -m uv add shap") from exc
from transformers import pipeline as hf_pipeline
from IPython.display import Image

ORDEN_TIPOS = ["VP", "VN", "FP", "FN"]
TOKENS_IGNORADOS = {"", "[CLS]", "[SEP]", "[PAD]", "<s>", "</s>", "<pad>"}


def normalizar_token(token):
    return str(token).replace("Ġ", " ").replace("##", "").strip().lower()


def explicar_variante(variante):
    pred = PRED_V7[variante]
    muestra = pd.concat(
        [
            pred[pred["tipo"] == tipo].sample(min(int((pred["tipo"] == tipo).sum()), N_SHAP_POR_TIPO), random_state=SEED)
            for tipo in ORDEN_TIPOS
        ],
        ignore_index=True,
    )
    ruta = str(MODELOS_DIR / variante)
    clasificador = hf_pipeline(
        "text-classification",
        model=ruta,
        tokenizer=ruta,
        top_k=None,
        truncation=True,
        max_length=MAX_LENGTH,
        device=0 if torch.cuda.is_available() else -1,
    )
    explicador = shap.Explainer(clasificador)
    valores = explicador(muestra["texto"].tolist(), batch_size=SHAP_BATCH)
    nombres = list(getattr(valores, "output_names", None) or [])
    k_ironia = nombres.index(ETIQUETAS[1]) if ETIQUETAS[1] in nombres else 1
    filas = []
    for i, fila in muestra.iterrows():
        tokens, contrib = valores.data[i], np.asarray(valores.values[i])[:, k_ironia]
        for pos, (tok, val) in enumerate(zip(tokens, contrib)):
            if str(tok).strip() in TOKENS_IGNORADOS:
                continue
            filas.append(
                {
                    "variante": variante,
                    "ID": fila["ID"],
                    "tipo": fila["tipo"],
                    "y_true": int(fila["y_true"]),
                    "y_pred": int(fila["y_pred"]),
                    "prob_ironia": float(fila["prob_ironia"]),
                    "posicion": pos,
                    "token": str(tok),
                    "token_normalizado": normalizar_token(tok),
                    "shap": float(val),
                }
            )
    del clasificador, explicador
    liberar_memoria()
    return valores, muestra, pd.DataFrame(filas)


SHAP_VALORES, SHAP_MUESTRAS, partes = {}, {}, []
monitor_shap = Monitor(RUN_DIR, total=len(VARIANTES), unidad="Variantes explicadas con SHAP")
try:
    for variante in VARIANTES:
        monitor_shap.iniciar(f"{variante.upper()} | SHAP ({N_SHAP_POR_TIPO} tuits por tipo)")
        valores, muestra, tabla = explicar_variante(variante)
        SHAP_VALORES[variante], SHAP_MUESTRAS[variante] = valores, muestra
        partes.append(tabla)
        monitor_shap.fit_terminado(
            f"{variante.upper()}: {muestra['ID'].nunique()} tuits explicados "
            f"({muestra['tipo'].value_counts().to_dict()})"
        )
except Exception as exc:
    monitor_shap.fallar(f"SHAP con error {type(exc).__name__}: {exc}")
    raise
monitor_shap.finalizar("SHAP completado")

SHAP_TOKENS = pd.concat(partes, ignore_index=True)
SHAP_TOKENS = SHAP_TOKENS[SHAP_TOKENS["token_normalizado"] != ""]
csv_atomico(SHAP_TOKENS, RESULTS_DIR / f"{PREFIJO}_v7_shap_tokens.csv")

SHAP_GLOBAL = (
    SHAP_TOKENS.groupby(["variante", "token_normalizado"])
    .agg(
        n_textos=("ID", "nunique"),
        shap_medio=("shap", "mean"),
        shap_abs_medio=("shap", lambda s: float(np.abs(s).mean())),
        shap_total=("shap", "sum"),
    )
    .reset_index()
)
SHAP_GLOBAL = SHAP_GLOBAL[SHAP_GLOBAL["n_textos"] >= MIN_TEXTOS_TOKEN].sort_values(
    ["variante", "shap_medio"], ascending=[True, False]
)
csv_atomico(SHAP_GLOBAL, RESULTS_DIR / f"{PREFIJO}_v7_shap_global.csv")
print("Tokens explicados por variante:", SHAP_TOKENS.groupby("variante")["ID"].nunique().to_dict())

### 11.1 Tokens más influyentes por variante (análisis global)

Barras rojas: tokens que en promedio empujan hacia **irónico**; azules: hacia **no irónico**. Solo tokens presentes en al menos `MIN_TEXTOS_TOKEN` tuits explicados, para no sobreinterpretar palabras raras.

In [ ]:
fig, axes = plt.subplots(1, len(VARIANTES), figsize=(5.4 * len(VARIANTES), 6.5), squeeze=False)
for ax, variante in zip(axes[0], VARIANTES):
    g = SHAP_GLOBAL[SHAP_GLOBAL["variante"] == variante]
    sel = pd.concat([g.nlargest(TOP_TOKENS, "shap_medio"), g.nsmallest(TOP_TOKENS, "shap_medio")])
    sel = sel.drop_duplicates("token_normalizado").sort_values("shap_medio")
    colores = ["#c0392b" if x > 0 else "#2e6da4" for x in sel["shap_medio"]]
    ax.barh(range(len(sel)), sel["shap_medio"], color=colores)
    ax.set_yticks(range(len(sel)), [f"{t}  (n={n})" for t, n in zip(sel["token_normalizado"], sel["n_textos"])], fontsize=8)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(variante.upper())
    ax.set_xlabel("Contribución SHAP media a «irónico»")
    ax.grid(axis="x", alpha=0.25)
fig.suptitle(f"{NOMBRE_EXPERIMENTO} (v7) — tokens más influyentes por variante (SHAP)", fontsize=12)
fig.tight_layout()
fig.savefig(RESULTS_DIR / f"{PREFIJO}_v7_shap_global.png", dpi=200, bbox_inches="tight")
fig.savefig(RESULTS_DIR / f"{PREFIJO}_v7_shap_global.pdf", bbox_inches="tight")
display(Image(filename=str(RESULTS_DIR / f"{PREFIJO}_v7_shap_global.png")))
plt.close(fig)

### 11.2 Ejemplos locales (VP, FP y FN)

Por variante se muestra el verdadero positivo y el falso positivo con **mayor** probabilidad de ironía y el falso negativo con **menor** probabilidad (los casos en que el modelo estuvo más seguro). Cada barra es un token del tuit, en el orden en que aparece. Además se guarda la vista coloreada de SHAP en `shap_html/<variante>_<tipo>.html` (se abre con el navegador).

In [ ]:
HTML_DIR = RESULTS_DIR / "shap_html"
HTML_DIR.mkdir(parents=True, exist_ok=True)
EJEMPLOS = [("VP", False), ("FP", False), ("FN", True)]
fig, axes = plt.subplots(len(VARIANTES), len(EJEMPLOS), figsize=(6.0 * len(EJEMPLOS), 3.6 * len(VARIANTES)), squeeze=False)
filas_ej = []
for i, variante in enumerate(VARIANTES):
    muestra = SHAP_MUESTRAS[variante]
    for j, (tipo, ascendente) in enumerate(EJEMPLOS):
        ax = axes[i, j]
        cand = muestra[muestra["tipo"] == tipo].sort_values("prob_ironia", ascending=ascendente)
        if cand.empty:
            ax.axis("off")
            continue
        k = cand.index[0]
        fila = cand.iloc[0]
        t = SHAP_TOKENS[(SHAP_TOKENS["variante"] == variante) & (SHAP_TOKENS["ID"] == fila["ID"])].sort_values("posicion")
        t = t.head(MAX_TOKENS_EJEMPLO)
        ax.bar(range(len(t)), t["shap"], color=["#c0392b" if x > 0 else "#2e6da4" for x in t["shap"]])
        ax.set_xticks(range(len(t)), t["token"].str.strip(), rotation=70, fontsize=7)
        ax.axhline(0, color="black", lw=0.8)
        ax.set_title(f"{variante.upper()} · {tipo} · P(irónico)={fila['prob_ironia']:.2f}", fontsize=9)
        if j == 0:
            ax.set_ylabel("SHAP hacia «irónico»")
        filas_ej.append({"variante": variante, "tipo": tipo, "ID": fila["ID"], "prob_ironia": fila["prob_ironia"],
                         "texto": fila["texto"], "mensaje_original": fila["mensaje_original"]})
        try:
            html_txt = shap.plots.text(SHAP_VALORES[variante][k], display=False)
            (HTML_DIR / f"{variante}_{tipo}.html").write_text(str(html_txt), encoding="utf-8")
        except Exception as exc:
            print(f"Aviso: no se pudo guardar el HTML de {variante}/{tipo}: {exc}")
fig.suptitle(f"{NOMBRE_EXPERIMENTO} (v7) — ejemplos explicados con SHAP", fontsize=12)
fig.tight_layout()
fig.savefig(RESULTS_DIR / f"{PREFIJO}_v7_shap_ejemplos.png", dpi=200, bbox_inches="tight")
fig.savefig(RESULTS_DIR / f"{PREFIJO}_v7_shap_ejemplos.pdf", bbox_inches="tight")
display(Image(filename=str(RESULTS_DIR / f"{PREFIJO}_v7_shap_ejemplos.png")))
plt.close(fig)
EJEMPLOS_TABLA = pd.DataFrame(filas_ej)
csv_atomico(EJEMPLOS_TABLA, RESULTS_DIR / f"{PREFIJO}_v7_shap_ejemplos.csv")
display(EJEMPLOS_TABLA[["variante", "tipo", "prob_ironia", "mensaje_original"]])

## 12. Resumen de la v7 y archivos generados

1. **No se repitió** la Nested CV, Optuna ni el test de 5 semillas: los resultados oficiales siguen siendo los de la v6.
2. Se entrenó y **exportó** un modelo final por variante (semilla 42, configuración v6) en `modelo_final/<variante>/`, con su `LEEME.md` de carga y `metadatos_v7.json`.
3. Se verificó que el modelo exportado reproduce la semilla 42 de la v6 (`<modelo>_v7_verificacion_exportacion.csv`).
4. Se explicaron con SHAP una muestra estratificada de tuits del test por variante (VP, VN, FP, FN): análisis global y ejemplos locales. Esta explicación no cambia ninguna decisión del modelo.

Archivos en la carpeta `_v7`: `modelo_final/`, `<modelo>_v7_verificacion_exportacion.csv`, `<modelo>_v7_test_predicciones.csv`, `<modelo>_v7_shap_tokens.csv`, `<modelo>_v7_shap_global.csv/.png/.pdf`, `<modelo>_v7_shap_ejemplos.csv/.png/.pdf` y `shap_html/`.

**Referencias:** Lundberg, S. M., y Lee, S.-I. (2017). A unified approach to interpreting model predictions. *NeurIPS 30*. — Ortega-Bueno, R., et al. (2022). Multi-view informed attention-based model for irony and satire detection in Spanish variants. *Knowledge-Based Systems*.